In [ ]:
import CDutils
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import savemat
import os
from sklearn.metrics import f1_score

In [ ]:
path = r"D:\Contraction_detection_publication\data"
dir_list = os.listdir(path)

In [ ]:
dir_list[0]

In [ ]:
f1_list = []
cp_score_list = []
cpn_list = []
for pt in range(12):
    #Load data
    data = sio.loadmat(path + '/' + dir_list[pt])
    print(dir_list[pt])
    modulation_ds = data['tmp_y']
    label_ds = data['newClusterSig']
    connectivity = data['Uterus']['tri'][0][0]
    conn = connectivity.astype(int)-1

    # modulation_ds = modulation[:,::40]
    # label_ds = label[:,::40]
    ch_num = modulation_ds.shape[0]
    conn_num = conn.shape[0]
    signal_length = modulation_ds.shape[1]
    
    #Spike artifact removal
    modulation_ar_removal = CDutils.ar_remove_patient(modulation_ds)
    #Matrix profile generation and estimate change points number
    profile = CDutils.matrix_profile(modulation_ar_removal,40)
    cp_num = CDutils.change_point_num(profile,t=0.49,d=200)

    modulation_ar_removal_t = np.transpose(modulation_ar_removal)
    segmentation = np.zeros_like(modulation_ar_removal_t)
    # Predict contraction location
    for i in range(conn_num):
        sig = modulation_ar_removal_t[:,conn[i,:].astype(int)]
        c = CDutils.contraction_detection(sig,cp_num)
        
        for j in range(3):
            segmentation[:,conn[i,j]]=segmentation[:,conn[i,j]]+c
   
    unique_values, counts = np.unique(conn, return_counts=True)
    # if dir_list[pt] == 'EMMI_111.mat':
    #     bi = segmentation==(counts)    
    # else:
    bi = segmentation>(counts/2)
    c_detection = np.transpose(bi)

    #Generate results
    f1_array = np.zeros(ch_num,)
    cp_score_array = np.zeros(ch_num,)
    cpn_array = np.zeros(ch_num,)
    for ch in range(ch_num):
        pre_cpds = CDutils.find_cpd(c_detection[ch,:], 1)
        gt_cpds = CDutils.find_cpd(label_ds[ch,:], 1)
        f1_array[ch] = f1_score(label_ds[ch,:], c_detection[ch,:], zero_division=1.0)
        cp_score_array[ch] = CDutils.cp_score(gt_cpds,pre_cpds,signal_length)
        
        cpn_array[ch] = (np.abs(len(gt_cpds)-len(pre_cpds)))/len(gt_cpds)
    print(np.median(f1_array))
    print(np.median(cp_score_array))
    print(np.median(cpn_array))
    f1_list.append(np.median(f1_array))
    cp_score_list.append(np.median(cp_score_array))
    cpn_list.append(np.median(cpn_array))
    

In [ ]:
mdic = {"f1_list": f1_list,"cp_score_list": cp_score_list,"cpn_list": cpn_list}
savemat(r"1_CD_result.mat", mdic)